<a href="https://colab.research.google.com/github/MohdAhmed627/Neural_Networks_for_Regression/blob/main/Energy_consumption.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ANN For Electricity Consumption Prediction Project**

**Overview**

This dataset has been created for developing Machine Learning models that predict monthly household electricity consumption.

The dataset contains various household characteristics including appliance usage, environmental conditions, occupancy details, and electricity tariff information.

It is suitable for regression-based machine learning projects, data analysis, academic research, and predictive analytics.

**License**

This project is intended for educational purposes, machine learning practice, and academic research.

In [34]:
import numpy as np
import pandas as pd
import tensorflow as tf
import keras

In [35]:
from google.colab import files
import pandas as pd
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_excel(file_name)

Saving electricity_consumption_dataset_20000.xlsx to electricity_consumption_dataset_20000 (10).xlsx


In [36]:
df.head()

,House_ID,House_Type,Family_Members,House_Area_sqft,Air_Conditioners,Refrigerators,Washing_Machines,Televisions,Computers,Water_Heaters,Ceiling_Fans,LED_Lights,EV_Charger,Solar_Panels,Daily_Usage_Hours,Average_Temperature,Season,Occupancy_Rate,Electricity_Tariff,Monthly_Electricity_Consumption
0,1001,Independent,2,602,2,1,0,2,0,0,6,32,Yes,Yes,5.9,36.1,Summer,81,8.01,1009.93
1,1002,Villa,5,3815,0,1,2,4,2,2,3,18,No,Yes,5.9,34.3,Summer,42,8.15,1163.77
2,1003,Villa,2,2761,2,3,2,3,4,1,7,9,Yes,Yes,19.5,40.3,Summer,69,7.68,1765.47
3,1004,Villa,6,1358,2,3,2,1,4,1,6,20,Yes,No,11.6,26.6,Rainy,89,8.38,1594.32
4,1005,Apartment,6,2143,1,1,0,3,1,3,5,34,Yes,No,6.8,28.7,Rainy,97,7.42,1392.67


In [37]:
df.duplicated().sum()

np.int64(0)

In [38]:
df.dtypes

,0
House_ID,int64
House_Type,object
Family_Members,int64
House_Area_sqft,int64
Air_Conditioners,int64
Refrigerators,int64
Washing_Machines,int64
Televisions,int64
Computers,int64
Water_Heaters,int64


In [39]:
df.columns.tolist()

['House_ID',
 'House_Type',
 'Family_Members',
 'House_Area_sqft',
 'Air_Conditioners',
 'Refrigerators',
 'Washing_Machines',
 'Televisions',
 'Computers',
 'Water_Heaters',
 'Ceiling_Fans',
 'LED_Lights',
 'EV_Charger',
 'Solar_Panels',
 'Daily_Usage_Hours',
 'Average_Temperature',
 'Season',
 'Occupancy_Rate',
 'Electricity_Tariff',
 'Monthly_Electricity_Consumption']

In [40]:
df['Occupancy_Rate'].value_counts()

,count
Occupancy_Rate,
78,368
74,359
79,356
71,354
68,350
...,...
41,306
52,305
87,305


In [41]:
df['House_Type'].value_counts()

,count
House_Type,
Villa,6701
Independent,6669
Apartment,6630


In [42]:
df.isnull().sum()

,0
House_ID,0
House_Type,0
Family_Members,0
House_Area_sqft,0
Air_Conditioners,0
Refrigerators,0
Washing_Machines,0
Televisions,0
Computers,0
Water_Heaters,0


In [43]:
df = df.drop(columns = ["House_ID","House_Type", "House_Area_sqft","Occupancy_Rate"] )

In [44]:
df

,Family_Members,Air_Conditioners,Refrigerators,Washing_Machines,Televisions,Computers,Water_Heaters,Ceiling_Fans,LED_Lights,EV_Charger,Solar_Panels,Daily_Usage_Hours,Average_Temperature,Season,Electricity_Tariff,Monthly_Electricity_Consumption
0,2,2,1,0,2,0,0,6,32,Yes,Yes,5.9,36.1,Summer,8.01,1009.93
1,5,0,1,2,4,2,2,3,18,No,Yes,5.9,34.3,Summer,8.15,1163.77
2,2,2,3,2,3,4,1,7,9,Yes,Yes,19.5,40.3,Summer,7.68,1765.47
3,6,2,3,2,1,4,1,6,20,Yes,No,11.6,26.6,Rainy,8.38,1594.32
4,6,1,1,0,3,1,3,5,34,Yes,No,6.8,28.7,Rainy,7.42,1392.67
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,4,4,2,0,3,2,1,5,23,Yes,No,22.8,33.5,Summer,7.35,1998.45
19996,1,3,3,1,1,2,1,5,17,No,Yes,21.3,26.6,Winter,5.79,1375.08
19997,8,0,1,1,4,3,1,8,17,Yes,Yes,7.2,25.9,Rainy,4.55,1048.03
19998,4,1,1,1,2,2,2,5,39,No,No,12.6,27.1,Rainy,8.62,1267.70


In [45]:
binary_cols = ["EV_Charger", "Solar_Panels"]

for col in binary_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

In [46]:
df

,Family_Members,Air_Conditioners,Refrigerators,Washing_Machines,Televisions,Computers,Water_Heaters,Ceiling_Fans,LED_Lights,EV_Charger,Solar_Panels,Daily_Usage_Hours,Average_Temperature,Season,Electricity_Tariff,Monthly_Electricity_Consumption
0,2,2,1,0,2,0,0,6,32,1,1,5.9,36.1,Summer,8.01,1009.93
1,5,0,1,2,4,2,2,3,18,0,1,5.9,34.3,Summer,8.15,1163.77
2,2,2,3,2,3,4,1,7,9,1,1,19.5,40.3,Summer,7.68,1765.47
3,6,2,3,2,1,4,1,6,20,1,0,11.6,26.6,Rainy,8.38,1594.32
4,6,1,1,0,3,1,3,5,34,1,0,6.8,28.7,Rainy,7.42,1392.67
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,4,4,2,0,3,2,1,5,23,1,0,22.8,33.5,Summer,7.35,1998.45
19996,1,3,3,1,1,2,1,5,17,0,1,21.3,26.6,Winter,5.79,1375.08
19997,8,0,1,1,4,3,1,8,17,1,1,7.2,25.9,Rainy,4.55,1048.03
19998,4,1,1,1,2,2,2,5,39,0,0,12.6,27.1,Rainy,8.62,1267.70


In [47]:
df['Season'].value_counts()

,count
Season,
Winter,6752
Rainy,6671
Summer,6577


In [48]:
df = pd.get_dummies(df, columns=["Season"], drop_first=True, dtype=int)

In [49]:
df

,Family_Members,Air_Conditioners,Refrigerators,Washing_Machines,Televisions,Computers,Water_Heaters,Ceiling_Fans,LED_Lights,EV_Charger,Solar_Panels,Daily_Usage_Hours,Average_Temperature,Electricity_Tariff,Monthly_Electricity_Consumption,Season_Summer,Season_Winter
0,2,2,1,0,2,0,0,6,32,1,1,5.9,36.1,8.01,1009.93,1,0
1,5,0,1,2,4,2,2,3,18,0,1,5.9,34.3,8.15,1163.77,1,0
2,2,2,3,2,3,4,1,7,9,1,1,19.5,40.3,7.68,1765.47,1,0
3,6,2,3,2,1,4,1,6,20,1,0,11.6,26.6,8.38,1594.32,0,0
4,6,1,1,0,3,1,3,5,34,1,0,6.8,28.7,7.42,1392.67,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,4,4,2,0,3,2,1,5,23,1,0,22.8,33.5,7.35,1998.45,1,0
19996,1,3,3,1,1,2,1,5,17,0,1,21.3,26.6,5.79,1375.08,0,1
19997,8,0,1,1,4,3,1,8,17,1,1,7.2,25.9,4.55,1048.03,0,0
19998,4,1,1,1,2,2,2,5,39,0,0,12.6,27.1,8.62,1267.70,0,0


In [50]:
x = df.drop(columns = "Monthly_Electricity_Consumption")
y = df["Monthly_Electricity_Consumption"]

In [51]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.2, random_state = 0)

In [52]:
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [53]:
from keras.layers import Dense,Input, Dropout

The Below is the first basic model

In [54]:
"""model = Sequential()
model.add(Input(shape=(16,)))
model.add(Dense(units = 32 , activation = "relu", kernel_initializer = "uniform"))
model.add(Dropout(0.2))
model.add(Dense(units = 64, activation = "relu", kernel_initializer = "uniform"))
model.add(Dropout(0.2))
model.add(Dense(units = 64, activation = "relu", kernel_initializer = "uniform"))
model.add(Dropout(0.2))
model.add(Dense(units = 128, activation = "relu", kernel_initializer = "uniform"))
model.add(Dropout(0.2))
model.add(Dense(units = 128, activation = "relu", kernel_initializer = "uniform"))
model.add(Dropout(0.2))
model.add(Dense(units = 100, activation = "relu", kernel_initializer = "uniform"))
model.add(Dropout(0.2))
model.add(Dense(
    units=1,
    activation="relu",
    kernel_initializer="uniform"
))

model.compile(
    optimizer="adam",
    loss="mean_squared_error",
    metrics = ["mae"]
) """


'model = Sequential()\nmodel.add(Input(shape=(16,)))\nmodel.add(Dense(units = 32 , activation = "relu", kernel_initializer = "uniform"))\nmodel.add(Dropout(0.2))\nmodel.add(Dense(units = 64, activation = "relu", kernel_initializer = "uniform"))\nmodel.add(Dropout(0.2))\nmodel.add(Dense(units = 64, activation = "relu", kernel_initializer = "uniform"))\nmodel.add(Dropout(0.2))\nmodel.add(Dense(units = 128, activation = "relu", kernel_initializer = "uniform"))\nmodel.add(Dropout(0.2))\nmodel.add(Dense(units = 128, activation = "relu", kernel_initializer = "uniform"))\nmodel.add(Dropout(0.2))\nmodel.add(Dense(units = 100, activation = "relu", kernel_initializer = "uniform"))\nmodel.add(Dropout(0.2))\nmodel.add(Dense(\n    units=1,\n    activation="relu",\n    kernel_initializer="uniform"\n))\n\nmodel.compile(\n    optimizer="adam",\n    loss="mean_squared_error",\n    metrics = ["mae"]\n) '

In [55]:
# model.fit(x_train, y_train, batch_size = 32, epochs = 150 )

In [56]:
# y_pred_train = model.predict(x_train)
# y_pred_test = model.predict(x_test)


In [57]:
from sklearn.metrics import accuracy_score, mean_squared_error

In [58]:
# print("MSE of train data:", mean_squared_error(y_train, y_pred_train))
# print("MSE of test data:", mean_squared_error(y_test,y_pred_test))

In the below ANN we have optimized it

In [59]:
from keras.layers import Dense, Dropout, Input, LeakyReLU
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam

# 1. Optimized Model Architecture
model = Sequential([
    Input(shape=(x_train.shape[1],)),
    Dense(128, kernel_initializer='he_normal'),
    LeakyReLU(negative_slope=0.01),
    Dropout(0.2),
    Dense(64, kernel_initializer='he_normal'),
    LeakyReLU(negative_slope=0.01),
    Dropout(0.2),
    Dense(64, kernel_initializer='he_normal'),
    LeakyReLU(negative_slope=0.01),
    Dropout(0.2),
    Dense(32, kernel_initializer='he_normal'),
    LeakyReLU(negative_slope=0.01),
    Dense(1, activation='relu') # Linear for regression
])

# 2. Advanced Compilation
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='huber', # More robust to outliers than MSE
    metrics=['mae']
)

# 3. Training Callbacks for maximum accuracy
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

# 4. Fit with validation split
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    batch_size=32,
    epochs=200,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 545.1172 - mae: 545.6169 - val_loss: 139.8251 - val_mae: 140.3248 - learning_rate: 0.0010
Epoch 2/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 179.6535 - mae: 180.1531 - val_loss: 128.0966 - val_mae: 128.5957 - learning_rate: 0.0010
Epoch 3/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 170.4570 - mae: 170.9565 - val_loss: 126.9025 - val_mae: 127.4020 - learning_rate: 0.0010
Epoch 4/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 167.7366 - mae: 168.2360 - val_loss: 118.5706 - val_mae: 119.0698 - learning_rate: 0.0010
Epoch 5/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 164.1263 - mae: 164.6258 - val_loss: 125.4542 - val_mae: 125.9533 - learning_rate: 0.0010
Epoch 6/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 163.1354 - mae: 163.6349 - val_loss: 114.4788 - val_mae: 114.9781 - learning_rate: 0.0010
Epoch 7/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 158.8132 - mae: 159.3127 - val_lo

In [60]:
y_pred_train = model.predict(x_train)
y_pred_test = model.predict(x_test)

print("MSE of train data:", mean_squared_error(y_train, y_pred_train))
print("MSE of test data:", mean_squared_error(y_test, y_pred_test))

500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
MSE of train data: 12727.284061845347
MSE of test data: 13407.82682102266


In [61]:
import pandas as pd

# Creating new data with the exact same columns used during training
new_data = pd.DataFrame({
    'Family_Members': [4],
    'Air_Conditioners': [2],
    'Refrigerators': [2],
    'Washing_Machines': [1],
    'Televisions': [2],
    'Computers': [2],
    'Water_Heaters': [1],
    'Ceiling_Fans': [6],
    'LED_Lights': [25],
    'EV_Charger': [0],
    'Solar_Panels': [1],
    'Daily_Usage_Hours': [10],
    'Average_Temperature': [35],
    'Electricity_Tariff': [7.5],
    'Season_Summer': [1], # One-hot encoded feature
    'Season_Winter': [0]  # Missing feature fixed
})

In [62]:
new_data_scaled = sc.transform(new_data)

In [63]:
sc.fit_transform(new_data)

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [64]:
# Re-scaling and predicting with the optimized model
new_data_scaled = sc.transform(new_data)
prediction = model.predict(new_data_scaled)

print(f"Optimized Predicted Monthly Consumption: {prediction[0][0]:.2f} units")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 229ms/step
Optimized Predicted Monthly Consumption: 1312.70 units
